## Assignment 2 Lab Question

In [1]:
# Two source signals are given:
# S = [[1, 2],
#      [2, 1],
#      [3, 2],
#      [4, 1]]
#
# Mixing matrix:
# A = [[1, 2],
#      [2, 1]]
#
# a) Generate mixed signals (X = AS)
# b) Apply ICA to recover source signals
# c) Compare original and recovered signals


import numpy as np
from sklearn.decomposition import FastICA

# Step 1: Define source signals (transpose for matrix multiplication)
S = np.array([
    [1, 2, 3, 4],
    [2, 1, 2, 1]
])

# Step 2: Define mixing matrix
A = np.array([
    [1, 2],
    [2, 1]
])

# Step 3: Generate mixed signals
X = np.dot(A, S)

print("Mixed Signals (X = AS):")
print(X)

# Step 4: Apply ICA
# sklearn expects samples as rows, so transpose
ica = FastICA(n_components=2, random_state=0)

S_estimated = ica.fit_transform(X.T)   # Recover signals
S_estimated = S_estimated.T            # Transpose back

print("\nRecovered Source Signals (after ICA):")
print(S_estimated)

# Step 5: Compare original and recovered signals
print("\nOriginal Source Signals:")
print(S)

print("\nRecovered Source Signals (approximate):")
print(np.round(S_estimated, 2))

Mixed Signals (X = AS):
[[5 4 7 6]
 [4 5 8 9]]

Recovered Source Signals (after ICA):
[[-1.00000066 -0.99999934  0.99999934  1.00000066]
 [ 0.99999934 -1.00000066  1.00000066 -0.99999934]]

Original Source Signals:
[[1 2 3 4]
 [2 1 2 1]]

Recovered Source Signals (approximate):
[[-1. -1.  1.  1.]
 [ 1. -1.  1. -1.]]


In [2]:
# Dataset:
# [1 2 3 4 5
#  2 3 4 5 6
#  3 4 5 6 7]
#
# a) Build autoencoder with architecture 5 → 2 → 5
# b) Train model to reconstruct input
# c) Print encoded (compressed) representation

import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# Step 1: Define dataset
X = np.array([
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
    [3, 4, 5, 6, 7]
], dtype=float)

# Step 2: Normalize data (helps training)
X = X / np.max(X)

# Step 3: Define Autoencoder architecture (5 → 2 → 5)
input_layer = Input(shape=(5,))

# Encoder
encoded = Dense(2, activation='relu')(input_layer)

# Decoder
decoded = Dense(5, activation='sigmoid')(encoded)

# Autoencoder model
autoencoder = Model(input_layer, decoded)

# Encoder-only model (for compressed representation)
encoder_model = Model(input_layer, encoded)

# Step 4: Compile model
autoencoder.compile(optimizer='adam', loss='mse')

# Step 5: Train model
autoencoder.fit(X, X, epochs=200, verbose=0)

# Step 6: Reconstructed output
reconstructed = autoencoder.predict(X)

print("Reconstructed Output:")
print(np.round(reconstructed, 3))

# Step 7: Encoded (compressed) representation
encoded_output = encoder_model.predict(X)

print("\nEncoded (Compressed) Representation:")
print(np.round(encoded_output, 3))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Reconstructed Output:
[[0.453 0.455 0.544 0.547 0.548]
 [0.453 0.455 0.544 0.547 0.548]
 [0.453 0.455 0.544 0.547 0.548]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step

Encoded (Compressed) Representation:
[[0. 0.]
 [0. 0.]
 [0. 0.]]


In [3]:
# 3: Active Learning Example
# Dataset: 120 samples, 2 features
# a) Start with 10 labeled samples
# b) Train classifier
# c) Select most uncertain sample
# d) Add to labeled set
# e) Repeat for 8 iterations

import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

# Step 1: Generate dataset
X, y = make_classification(n_samples=120,
                           n_features=2,
                           n_redundant=0,
                           n_informative=2,
                           n_clusters_per_class=1,
                           random_state=42)

# Step 2: Initial labeled dataset (first 10 samples)
labeled_indices = list(range(10))

# Remaining unlabeled pool
unlabeled_indices = list(range(10, 120))

# Step 3: Active learning loop (8 iterations)
model = LogisticRegression()

for iteration in range(8):

    # Train classifier
    model.fit(X[labeled_indices], y[labeled_indices])

    # Predict probabilities on unlabeled pool
    probs = model.predict_proba(X[unlabeled_indices])

    # Measure uncertainty (closest to 0.5)
    uncertainty = np.abs(probs[:, 0] - 0.5)

    # Select most uncertain sample
    query_index = np.argmin(uncertainty)

    selected_sample = unlabeled_indices[query_index]

    # Add to labeled dataset
    labeled_indices.append(selected_sample)

    # Remove from unlabeled pool
    unlabeled_indices.remove(selected_sample)

    print(f"Iteration {iteration+1}: Added sample index {selected_sample}")

# Final labeled dataset size
print("\nFinal labeled dataset size:", len(labeled_indices))

Iteration 1: Added sample index 116
Iteration 2: Added sample index 94
Iteration 3: Added sample index 108
Iteration 4: Added sample index 62
Iteration 5: Added sample index 36
Iteration 6: Added sample index 103
Iteration 7: Added sample index 33
Iteration 8: Added sample index 106

Final labeled dataset size: 18


In [4]:
# 4. Radial Basis Function (RBF)
# Given:
# X = [1, 3, 5]
# Centers = [2, 4]
# sigma = 1
#
# a) Compute Euclidean distance
# b) Apply Gaussian RBF
# c) Calculate hidden layer output

import numpy as np

# Step 1: Define inputs
X = np.array([1, 3, 5])
centers = np.array([2, 4])
sigma = 1

# Step 2: Compute Euclidean distance matrix
distance_matrix = np.zeros((len(X), len(centers)))

for i in range(len(X)):
    for j in range(len(centers)):
        distance_matrix[i][j] = abs(X[i] - centers[j])

print("Euclidean Distance Matrix:")
print(distance_matrix)

# Step 3: Apply Gaussian RBF
rbf_output = np.exp(-(distance_matrix**2) / (2 * sigma**2))

print("\nGaussian RBF Output (Hidden Layer Output):")
print(np.round(rbf_output, 4))

Euclidean Distance Matrix:
[[1. 3.]
 [1. 1.]
 [3. 1.]]

Gaussian RBF Output (Hidden Layer Output):
[[0.6065 0.0111]
 [0.6065 0.6065]
 [0.0111 0.6065]]


In [5]:
# 5. Apriori Algorithm
# Transactions:
# T1: {A, B}
# T2: {A, C}
# T3: {B, C}
# T4: {A, B, C}
#
# a) Generate candidate itemsets
# b) Compute support
# c) Find frequent itemsets (min support = 2)

from itertools import combinations

# Step 1: Define transactions
transactions = [
    {'A','B'},
    {'A','C'},
    {'B','C'},
    {'A','B','C'}
]

min_support = 2

# Step 2: Function to calculate support
def get_support(itemset, transactions):
    count = 0
    for t in transactions:
        if itemset.issubset(t):
            count += 1
    return count


# --------------------------
# Step 3: Candidate 1-itemsets (C1)
# --------------------------
items = set().union(*transactions)
C1 = [{i} for i in items]

print("C1 Candidates:")
print(C1)

# Compute support for C1
L1 = []
print("\nSupport Count for C1:")
for item in C1:
    support = get_support(item, transactions)
    print(item, ":", support)
    if support >= min_support:
        L1.append(item)

print("\nFrequent Itemsets L1:")
print(L1)


# --------------------------
# Step 4: Candidate 2-itemsets (C2)
# --------------------------
C2 = [set(i) for i in combinations(items, 2)]

print("\nC2 Candidates:")
print(C2)

L2 = []
print("\nSupport Count for C2:")
for item in C2:
    support = get_support(item, transactions)
    print(item, ":", support)
    if support >= min_support:
        L2.append(item)

print("\nFrequent Itemsets L2:")
print(L2)


# --------------------------
# Step 5: Candidate 3-itemsets (C3)
# --------------------------
C3 = [set(i) for i in combinations(items, 3)]

print("\nC3 Candidates:")
print(C3)

L3 = []
print("\nSupport Count for C3:")
for item in C3:
    support = get_support(item, transactions)
    print(item, ":", support)
    if support >= min_support:
        L3.append(item)

print("\nFrequent Itemsets L3:")
print(L3)

C1 Candidates:
[{'C'}, {'B'}, {'A'}]

Support Count for C1:
{'C'} : 3
{'B'} : 3
{'A'} : 3

Frequent Itemsets L1:
[{'C'}, {'B'}, {'A'}]

C2 Candidates:
[{'C', 'B'}, {'A', 'C'}, {'A', 'B'}]

Support Count for C2:
{'C', 'B'} : 2
{'A', 'C'} : 2
{'A', 'B'} : 2

Frequent Itemsets L2:
[{'C', 'B'}, {'A', 'C'}, {'A', 'B'}]

C3 Candidates:
[{'A', 'C', 'B'}]

Support Count for C3:
{'A', 'C', 'B'} : 1

Frequent Itemsets L3:
[]


In [6]:
# 6. ECLAT Algorithm
# Transactions:
# T1: {A, B}
# T2: {A, C}
# T3: {B, C}
#
# a) Convert to TID list
# b) Compute intersections
# c) Identify frequent itemsets (min support = 2)

from itertools import combinations

# Step 1: Define transactions with IDs
transactions = {
    "T1": {"A", "B"},
    "T2": {"A", "C"},
    "T3": {"B", "C"}
}

min_support = 2

# Step 2: Convert to TID list representation
tid_list = {}

for tid, items in transactions.items():
    for item in items:
        if item not in tid_list:
            tid_list[item] = set()
        tid_list[item].add(tid)

print("TID Lists:")
for item, tids in tid_list.items():
    print(item, ":", tids)


# Step 3: Compute intersections (2-itemsets)
print("\nIntersections (2-itemsets):")

frequent_itemsets = {}

items = list(tid_list.keys())

for pair in combinations(items, 2):
    intersection = tid_list[pair[0]].intersection(tid_list[pair[1]])
    print(pair, ":", intersection)

    if len(intersection) >= min_support:
        frequent_itemsets[pair] = intersection


# Step 4: Display frequent itemsets
print("\nFrequent Itemsets (min_support = 2):")

for itemset, tids in frequent_itemsets.items():
    print(itemset, "Support =", len(tids))

TID Lists:
A : {'T1', 'T2'}
B : {'T3', 'T1'}
C : {'T3', 'T2'}

Intersections (2-itemsets):
('A', 'B') : {'T1'}
('A', 'C') : {'T2'}
('B', 'C') : {'T3'}

Frequent Itemsets (min_support = 2):


In [20]:
# 7. Bagging Algorithm (Fixed Version)

import numpy as np
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from scipy.stats import mode

# Step 1: Generate dataset
X, y = make_classification(
    n_samples=400,
    n_features=4,
    n_informative=4,
    n_redundant=0,
    random_state=42
)

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Step 3: Bootstrap sampling + train 10 decision trees
n_models = 10
models = []

for i in range(n_models):

    indices = np.random.choice(len(X_train), len(X_train), replace=True)

    X_bootstrap = X_train[indices]
    y_bootstrap = y_train[indices]

    model = DecisionTreeClassifier()
    model.fit(X_bootstrap, y_bootstrap)

    models.append(model)

print("10 Decision Trees trained successfully")


# Step 4: Majority Voting
predictions = np.array([model.predict(X_test) for model in models])

# FIXED LINE
final_predictions = mode(predictions, axis=0, keepdims=False).mode


# Step 5: Compute accuracy
accuracy = accuracy_score(y_test, final_predictions)

print("\nFinal Accuracy after Bagging:", round(accuracy * 100, 2), "%")

10 Decision Trees trained successfully

Final Accuracy after Bagging: 91.67 %


In [8]:
# 10. Boosting Algorithm
# Dataset: 400 samples, 4 features
# a) Train boosting model
# b) Increase weights for misclassified samples
# c) Combine weak learners
# d) Compare performance with Bagging

import numpy as np
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


# Step 1: Generate dataset
X, y = make_classification(
    n_samples=400,
    n_features=4,
    n_informative=4,
    n_redundant=0,
    random_state=42
)

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


# -------------------------
# BAGGING MODEL
# -------------------------
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=10,
    random_state=42
)

bagging_model.fit(X_train, y_train)
bagging_predictions = bagging_model.predict(X_test)

bagging_accuracy = accuracy_score(y_test, bagging_predictions)


# -------------------------
# BOOSTING MODEL (AdaBoost)
# -------------------------
boosting_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # weak learner
    n_estimators=10,
    random_state=42
)

boosting_model.fit(X_train, y_train)
boosting_predictions = boosting_model.predict(X_test)

boosting_accuracy = accuracy_score(y_test, boosting_predictions)


# Step 3: Print results
print("Bagging Accuracy :", round(bagging_accuracy * 100, 2), "%")
print("Boosting Accuracy:", round(boosting_accuracy * 100, 2), "%")

Bagging Accuracy : 90.0 %
Boosting Accuracy: 75.83 %


In [9]:
# 11. Agent–Environment Interaction
# States: {0,1,2,3}
# Actions: {Left, Right}
# Reward: +1 for reaching goal state (3), otherwise 0
# Simulate agent movement for 10 steps
# Display state transitions

import random

# Step 1: Define environment
states = [0, 1, 2, 3]
actions = ["Left", "Right"]
goal_state = 3

# Step 2: Initial state
current_state = 0

print("Initial State:", current_state)
print("\nState Transitions:\n")

# Step 3: Simulate 10 steps
for step in range(10):

    action = random.choice(actions)

    # Apply action rules
    if action == "Left":
        next_state = max(0, current_state - 1)
    else:
        next_state = min(3, current_state + 1)

    # Assign reward
    reward = 1 if next_state == goal_state else 0

    # Display transition
    print(f"Step {step+1}: State {current_state} --{action}--> State {next_state}, Reward = {reward}")

    # Update state
    current_state = next_state

Initial State: 0

State Transitions:

Step 1: State 0 --Right--> State 1, Reward = 0
Step 2: State 1 --Left--> State 0, Reward = 0
Step 3: State 0 --Right--> State 1, Reward = 0
Step 4: State 1 --Left--> State 0, Reward = 0
Step 5: State 0 --Right--> State 1, Reward = 0
Step 6: State 1 --Left--> State 0, Reward = 0
Step 7: State 0 --Right--> State 1, Reward = 0
Step 8: State 1 --Left--> State 0, Reward = 0
Step 9: State 0 --Left--> State 0, Reward = 0
Step 10: State 0 --Left--> State 0, Reward = 0


In [10]:
# 12. Exploration vs Exploitation
# a) Implement ε-greedy strategy with ε = 0.2
# b) Simulate action selection for 20 iterations
# c) Count exploration vs exploitation steps
# d) Print results

import random
import numpy as np

# Step 1: Define parameters
epsilon = 0.2
iterations = 20

# Example action-value estimates (Q-values)
Q_values = np.array([0.5, 0.8, 0.3])  # assume 3 actions

actions = [0, 1, 2]

exploration_count = 0
exploitation_count = 0

print("Action Selection Process:\n")

# Step 2: Run simulation
for i in range(iterations):

    if random.random() < epsilon:
        # Exploration
        action = random.choice(actions)
        exploration_count += 1
        print(f"Iteration {i+1}: Exploration → Selected action {action}")

    else:
        # Exploitation
        action = np.argmax(Q_values)
        exploitation_count += 1
        print(f"Iteration {i+1}: Exploitation → Selected action {action}")

# Step 3: Print summary results
print("\nSummary Results 📊")
print("Total Iterations:", iterations)
print("Exploration Steps:", exploration_count)
print("Exploitation Steps:", exploitation_count)

Action Selection Process:

Iteration 1: Exploitation → Selected action 1
Iteration 2: Exploitation → Selected action 1
Iteration 3: Exploration → Selected action 2
Iteration 4: Exploitation → Selected action 1
Iteration 5: Exploitation → Selected action 1
Iteration 6: Exploration → Selected action 1
Iteration 7: Exploitation → Selected action 1
Iteration 8: Exploitation → Selected action 1
Iteration 9: Exploration → Selected action 1
Iteration 10: Exploitation → Selected action 1
Iteration 11: Exploitation → Selected action 1
Iteration 12: Exploitation → Selected action 1
Iteration 13: Exploration → Selected action 2
Iteration 14: Exploration → Selected action 0
Iteration 15: Exploitation → Selected action 1
Iteration 16: Exploitation → Selected action 1
Iteration 17: Exploitation → Selected action 1
Iteration 18: Exploitation → Selected action 1
Iteration 19: Exploration → Selected action 1
Iteration 20: Exploitation → Selected action 1

Summary Results 📊
Total Iterations: 20
Explorat

In [11]:
# 13. Policy Representation
# a) Define deterministic policy π(s) → a
# b) Convert into stochastic policy π(s) → p(a)
# c) Show action probabilities for each state

import numpy as np

# Step 1: Define states and actions
states = [0, 1, 2, 3]
actions = ["Left", "Right"]

# Step 2: Define deterministic policy
# Example: always move Right except last state
deterministic_policy = {
    0: "Right",
    1: "Right",
    2: "Right",
    3: "Left"
}

print("Deterministic Policy π(s) → a:\n")
for s in states:
    print(f"π({s}) = {deterministic_policy[s]}")


# Step 3: Convert deterministic policy into stochastic policy
# Rule: chosen action gets probability 0.8, other gets 0.2

stochastic_policy = {}

for s in states:
    stochastic_policy[s] = {}

    chosen_action = deterministic_policy[s]

    for a in actions:
        if a == chosen_action:
            stochastic_policy[s][a] = 0.8
        else:
            stochastic_policy[s][a] = 0.2


# Step 4: Display stochastic policy
print("\nStochastic Policy π(s) → p(a):\n")

for s in states:
    print(f"State {s}: {stochastic_policy[s]}")

Deterministic Policy π(s) → a:

π(0) = Right
π(1) = Right
π(2) = Right
π(3) = Left

Stochastic Policy π(s) → p(a):

State 0: {'Left': 0.2, 'Right': 0.8}
State 1: {'Left': 0.2, 'Right': 0.8}
State 2: {'Left': 0.2, 'Right': 0.8}
State 3: {'Left': 0.8, 'Right': 0.2}


In [12]:
# 14. Markov Chain
# Weather states: {Sunny, Rainy}
# Transition matrix:
# P = [[0.8, 0.2],
#      [0.4, 0.6]]
#
# a) Simulate weather for 10 days
# b) Count frequency of each state
# c) Verify transition probabilities

import numpy as np

# Step 1: Define states
states = ["Sunny", "Rainy"]

# Step 2: Transition matrix
P = np.array([
    [0.8, 0.2],  # Sunny → Sunny, Rainy
    [0.4, 0.6]   # Rainy → Sunny, Rainy
])

# Step 3: Initial state
current_state = 0  # Start with Sunny

days = 10
weather_sequence = [states[current_state]]

# Step 4: Simulate 10 days
for day in range(days - 1):

    current_state = np.random.choice([0, 1], p=P[current_state])
    weather_sequence.append(states[current_state])

print("Weather sequence for 10 days:")
print(weather_sequence)


# Step 5: Count frequency of each state
sunny_count = weather_sequence.count("Sunny")
rainy_count = weather_sequence.count("Rainy")

print("\nState Frequency:")
print("Sunny:", sunny_count)
print("Rainy:", rainy_count)


# Step 6: Verify transition probabilities from simulation
transition_counts = {
    "Sunny->Sunny": 0,
    "Sunny->Rainy": 0,
    "Rainy->Sunny": 0,
    "Rainy->Rainy": 0
}

for i in range(len(weather_sequence)-1):

    prev_state = weather_sequence[i]
    next_state = weather_sequence[i+1]

    transition_counts[f"{prev_state}->{next_state}"] += 1

print("\nObserved Transition Counts:")
for k, v in transition_counts.items():
    print(k, ":", v)

Weather sequence for 10 days:
['Sunny', 'Sunny', 'Rainy', 'Sunny', 'Sunny', 'Sunny', 'Sunny', 'Sunny', 'Sunny', 'Sunny']

State Frequency:
Sunny: 9
Rainy: 1

Observed Transition Counts:
Sunny->Sunny : 7
Sunny->Rainy : 1
Rainy->Sunny : 1
Rainy->Rainy : 0


In [13]:
# 15. Markov Reward Process
# States: {S1, S2}
# Rewards: R(S1)=5, R(S2)=2
# gamma = 0.9
#
# a) Define transition probabilities
# b) Compute value function
# c) Interpret results

import numpy as np

# Step 1: Define states
states = ["S1", "S2"]

# Step 2: Define rewards
R = np.array([5, 2])   # Reward vector

# Step 3: Define transition probability matrix
# Example:
# S1 -> S1 = 0.7, S1 -> S2 = 0.3
# S2 -> S1 = 0.4, S2 -> S2 = 0.6

P = np.array([
    [0.7, 0.3],
    [0.4, 0.6]
])

# Step 4: Discount factor
gamma = 0.9

# Step 5: Compute value function
# Formula: V = (I - gamma * P)^(-1) * R

I = np.eye(len(states))

V = np.linalg.inv(I - gamma * P).dot(R)

# Step 6: Display results
print("Value Function Results:\n")

for i in range(len(states)):
    print(f"V({states[i]}) =", round(V[i], 4))

Value Function Results:

V(S1) = 38.9041
V(S2) = 34.7945


In [14]:
# 16. Policy Evaluation
# States: {0,1,2}
# Actions: {Left, Right}
# a) Initialize V(s)=0
# b) Perform iterative policy evaluation
# c) Show updates until convergence

import numpy as np

# Step 1: Define states
states = [0, 1, 2]

# Step 2: Define parameters
gamma = 0.9
theta = 0.001  # convergence threshold

# Step 3: Initialize value function V(s)=0
V = np.zeros(len(states))

# Step 4: Define transition function
# Terminal state = 2

def step(state, action):
    if state == 2:
        return state, 0  # terminal

    if action == "Left":
        next_state = max(0, state - 1)
    else:
        next_state = min(2, state + 1)

    reward = 1 if next_state == 2 else 0
    return next_state, reward


# Step 5: Policy (uniform random)
actions = ["Left", "Right"]
policy_prob = 0.5


# Step 6: Iterative Policy Evaluation
iteration = 0

while True:

    delta = 0
    new_V = V.copy()

    print(f"\nIteration {iteration+1}")

    for s in states:

        if s == 2:
            continue  # terminal state

        v_old = V[s]
        v_new = 0

        for a in actions:
            next_state, reward = step(s, a)
            v_new += policy_prob * (reward + gamma * V[next_state])

        new_V[s] = v_new

        print(f"V({s}) updated from {round(v_old,4)} → {round(v_new,4)}")

        delta = max(delta, abs(v_old - v_new))

    V = new_V
    iteration += 1

    if delta < theta:
        break


# Step 7: Final Values
print("\nFinal Converged Value Function:")
for s in states:
    print(f"V({s}) = {round(V[s],4)}")


Iteration 1
V(0) updated from 0.0 → 0.0
V(1) updated from 0.0 → 0.5

Iteration 2
V(0) updated from 0.0 → 0.225
V(1) updated from 0.5 → 0.5

Iteration 3
V(0) updated from 0.225 → 0.3263
V(1) updated from 0.5 → 0.6013

Iteration 4
V(0) updated from 0.3263 → 0.4174
V(1) updated from 0.6013 → 0.6468

Iteration 5
V(0) updated from 0.4174 → 0.4789
V(1) updated from 0.6468 → 0.6878

Iteration 6
V(0) updated from 0.4789 → 0.525
V(1) updated from 0.6878 → 0.7155

Iteration 7
V(0) updated from 0.525 → 0.5582
V(1) updated from 0.7155 → 0.7363

Iteration 8
V(0) updated from 0.5582 → 0.5825
V(1) updated from 0.7363 → 0.7512

Iteration 9
V(0) updated from 0.5825 → 0.6002
V(1) updated from 0.7512 → 0.7621

Iteration 10
V(0) updated from 0.6002 → 0.613
V(1) updated from 0.7621 → 0.7701

Iteration 11
V(0) updated from 0.613 → 0.6224
V(1) updated from 0.7701 → 0.7759

Iteration 12
V(0) updated from 0.6224 → 0.6292
V(1) updated from 0.7759 → 0.7801

Iteration 13
V(0) updated from 0.6292 → 0.6342
V(1) up

In [15]:
# 17. Policy Improvement
# a) Use value function from Question 16
# b) Improve policy using greedy approach
# c) Show updated policy

import numpy as np

# Step 1: Define states and actions
states = [0, 1, 2]
actions = ["Left", "Right"]

gamma = 0.9

# Step 2: Value function from Question 16 (approximate converged values)
V = np.array([0.405, 0.9, 0.0])

# Step 3: Transition function (same as Q16)
def step(state, action):

    if state == 2:
        return state, 0

    if action == "Left":
        next_state = max(0, state - 1)
    else:
        next_state = min(2, state + 1)

    reward = 1 if next_state == 2 else 0

    return next_state, reward


# Step 4: Greedy Policy Improvement
policy = {}

print("Greedy Policy Improvement:\n")

for s in states:

    if s == 2:
        policy[s] = "Terminal"
        continue

    action_values = {}

    for a in actions:
        next_state, reward = step(s, a)

        action_values[a] = reward + gamma * V[next_state]

    best_action = max(action_values, key=action_values.get)

    policy[s] = best_action

    print(f"State {s}: {action_values} → Best Action = {best_action}")


# Step 5: Display updated policy
print("\nUpdated Greedy Policy π'(s):\n")

for s in states:
    print(f"π({s}) = {policy[s]}")

Greedy Policy Improvement:

State 0: {'Left': np.float64(0.36450000000000005), 'Right': np.float64(0.81)} → Best Action = Right
State 1: {'Left': np.float64(0.36450000000000005), 'Right': np.float64(1.0)} → Best Action = Right

Updated Greedy Policy π'(s):

π(0) = Right
π(1) = Right
π(2) = Terminal


In [16]:
# 18. Value Iteration
# a) Initialize V(s)=0
# b) Apply Bellman optimality update
# c) Run until convergence
# d) Extract optimal policy

import numpy as np

# Step 1: Define states and actions
states = [0, 1, 2]
actions = ["Left", "Right"]

gamma = 0.9
theta = 0.001  # convergence threshold

# Step 2: Initialize value function
V = np.zeros(len(states))


# Step 3: Transition function
def step(state, action):

    if state == 2:
        return state, 0  # terminal state

    if action == "Left":
        next_state = max(0, state - 1)
    else:
        next_state = min(2, state + 1)

    reward = 1 if next_state == 2 else 0

    return next_state, reward


# Step 4: Value Iteration Loop
iteration = 0

while True:

    delta = 0
    print(f"\nIteration {iteration+1}")

    for s in states:

        if s == 2:
            continue

        v_old = V[s]

        action_values = []

        for a in actions:
            next_state, reward = step(s, a)
            action_values.append(reward + gamma * V[next_state])

        V[s] = max(action_values)

        print(f"V({s}) updated from {round(v_old,4)} → {round(V[s],4)}")

        delta = max(delta, abs(v_old - V[s]))

    iteration += 1

    if delta < theta:
        break


# Step 5: Extract Optimal Policy
policy = {}

for s in states:

    if s == 2:
        policy[s] = "Terminal"
        continue

    action_values = {}

    for a in actions:
        next_state, reward = step(s, a)
        action_values[a] = reward + gamma * V[next_state]

    best_action = max(action_values, key=action_values.get)
    policy[s] = best_action


# Step 6: Final Results
print("\nFinal Optimal Value Function:")
for s in states:
    print(f"V({s}) = {round(V[s],4)}")

print("\nOptimal Policy:")
for s in states:
    print(f"π({s}) = {policy[s]}")


Iteration 1
V(0) updated from 0.0 → 0.0
V(1) updated from 0.0 → 1.0

Iteration 2
V(0) updated from 0.0 → 0.9
V(1) updated from 1.0 → 1.0

Iteration 3
V(0) updated from 0.9 → 0.9
V(1) updated from 1.0 → 1.0

Final Optimal Value Function:
V(0) = 0.9
V(1) = 1.0
V(2) = 0.0

Optimal Policy:
π(0) = Right
π(1) = Right
π(2) = Terminal


In [17]:
# 19. Monte Carlo Method
# a) Generate 5 episodes of state transitions
# b) Compute returns for each episode
# c) Estimate value function using average returns

import random
import numpy as np

# Step 1: Define environment
states = [0, 1, 2]
actions = ["Left", "Right"]
terminal_state = 2
gamma = 0.9

# Step 2: Transition function
def step(state, action):

    if state == terminal_state:
        return state, 0

    if action == "Left":
        next_state = max(0, state - 1)
    else:
        next_state = min(2, state + 1)

    reward = 1 if next_state == terminal_state else 0
    return next_state, reward


# Step 3: Generate episodes
num_episodes = 5
episodes = []

print("Generated Episodes:\n")

for ep in range(num_episodes):

    episode = []
    state = random.choice([0, 1])  # start randomly (non-terminal)

    while state != terminal_state:

        action = random.choice(actions)
        next_state, reward = step(state, action)

        episode.append((state, reward))
        state = next_state

    episodes.append(episode)

    print(f"Episode {ep+1}: {episode}")


# Step 4: Compute returns
returns = {0: [], 1: []}

print("\nReturns per Episode:\n")

for episode in episodes:

    G = 0

    for t in reversed(range(len(episode))):

        state, reward = episode[t]
        G = gamma * G + reward
        returns[state].append(G)

print("Returns:", returns)


# Step 5: Estimate Value Function
V = {}

for state in returns:
    if returns[state]:
        V[state] = np.mean(returns[state])
    else:
        V[state] = 0

V[2] = 0  # terminal state

print("\nEstimated Value Function:")
for s in states:
    print(f"V({s}) = {round(V[s],4)}")

Generated Episodes:

Episode 1: [(1, 1)]
Episode 2: [(1, 0), (0, 0), (1, 0), (0, 0), (0, 0), (0, 0), (1, 1)]
Episode 3: [(1, 0), (0, 0), (0, 0), (1, 1)]
Episode 4: [(0, 0), (0, 0), (1, 0), (0, 0), (1, 1)]
Episode 5: [(1, 1)]

Returns per Episode:

Returns: {0: [0.9, 0.81, 0.7290000000000001, 0.5904900000000002, 0.9, 0.81, 0.9, 0.7290000000000001, 0.6561000000000001], 1: [1.0, 1.0, 0.6561000000000001, 0.5314410000000002, 1.0, 0.7290000000000001, 1.0, 0.81, 1.0]}

Estimated Value Function:
V(0) = 0.7805
V(1) = 0.8585
V(2) = 0


In [18]:
# 20. Q-Learning
# States: {0,1,2}
# Actions: {Left, Right}
# alpha = 0.1
# gamma = 0.9
# Train for 20 episodes
# Print final Q-table

import numpy as np
import random

# Step 1: Define environment
states = [0, 1, 2]
actions = ["Left", "Right"]

alpha = 0.1   # learning rate
gamma = 0.9   # discount factor
epsilon = 0.2 # exploration probability

# Step 2: Initialize Q-table
Q = np.zeros((len(states), len(actions)))

terminal_state = 2


# Step 3: Transition function
def step(state, action):

    if state == terminal_state:
        return state, 0

    if action == "Left":
        next_state = max(0, state - 1)
    else:
        next_state = min(2, state + 1)

    reward = 1 if next_state == terminal_state else 0

    return next_state, reward


# Step 4: Q-learning training (20 episodes)
episodes = 20

for ep in range(episodes):

    state = random.choice([0, 1])  # start from non-terminal state

    while state != terminal_state:

        # ε-greedy action selection
        if random.random() < epsilon:
            action_index = random.choice([0, 1])
        else:
            action_index = np.argmax(Q[state])

        action = actions[action_index]

        next_state, reward = step(state, action)

        # Q-learning update rule
        Q[state][action_index] = Q[state][action_index] + alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state][action_index]
        )

        state = next_state


# Step 5: Print final Q-table
print("Final Q-table:\n")

for s in range(len(states)):
    print(f"State {s}: Left={round(Q[s][0],4)}, Right={round(Q[s][1],4)}")

Final Q-table:

State 0: Left=0.003, Right=0.4213
State 1: Left=0.0615, Right=0.8784
State 2: Left=0.0, Right=0.0
